# Preprocesamiento y Embeddings
## Motor de búsqueda semántica de ofertas laborales

**Proyecto Integrador 1 — Ingeniería de Sistemas**

### Objetivo del notebook

Continuar el flujo de trabajo a partir de las conclusiones del EDA (`01_EDA.ipynb`), cubriendo los tres puntos indicados por el tutor:

1. **Terminar la limpieza y el preprocesamiento** de los datos.
2. **Primera iteración de representación semántica** mediante *Sentence-BERT* (Sentence Transformers).
3. **Exploración de bases de datos vectoriales** (Pinecone y alternativas), comparándolas con FAISS (usado en el anteproyecto).

Este notebook asume que `01_EDA.ipynb` ya se ejecutó y que el dataset crudo está disponible en `/data/raw/job_descriptions.csv` (o se vuelve a descargar de Kaggle).

## 1. Importación de librerías

Para esta etapa se necesitan, además de Pandas/NumPy:

- **sentence-transformers**: para generar los embeddings (Sentence-BERT).
- **faiss-cpu**: índice vectorial local, usado como línea base (baseline) — es lo que se propuso en el anteproyecto.
- **re**: limpieza de texto con expresiones regulares.
- **tqdm**: barra de progreso al generar embeddings sobre muchos registros.

Instalación (en Colab, descomentar la primera vez):

In [ ]:
# !pip install -q sentence-transformers faiss-cpu tqdm pinecone-client chromadb


In [ ]:
import os
import re
import time

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)


## 2. Carga del dataset

**Importante en Colab:** el almacenamiento local (`/content/...`) es efímero — se borra cada vez que se reinicia o desconecta el runtime. Como este notebook suele abrirse en una sesión distinta a la del EDA, no se puede asumir que el CSV ya está en disco.

Para no tener que descargar 1.6M de registros (y volver a correr toda la limpieza) cada vez que se abre una sesión nueva, se monta Google Drive y se persisten ahí tanto el dataset crudo como los archivos procesados. Si el archivo ya existe en Drive (por ejemplo porque ya corriste este notebook antes o porque un compañero de equipo ya lo dejó ahí), se reutiliza; si no, se descarga de Kaggle.

In [ ]:
from pathlib import Path

MONTAR_DRIVE = True  # cambia a False si prefieres trabajar solo con el almacenamiento efimero de Colab

if MONTAR_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = Path("/content/drive/MyDrive/proyecto_integrador")
else:
    BASE_DIR = Path("/content")

RUTA_RAW = BASE_DIR / "data" / "raw"
RUTA_PROCESSED = BASE_DIR / "data" / "processed"
RUTA_RAW.mkdir(parents=True, exist_ok=True)
RUTA_PROCESSED.mkdir(parents=True, exist_ok=True)

RUTA_DATASET = RUTA_RAW / "job_descriptions.csv"
print(f"Dataset crudo esperado en: {RUTA_DATASET}")
print(f"Archivos procesados se guardaran en: {RUTA_PROCESSED}")


In [ ]:
if not RUTA_DATASET.exists():
    print("Dataset no encontrado, descargando desde Kaggle...")
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
    !kaggle datasets download -d ravindrasinghrana/job-description-dataset -p {RUTA_RAW} --unzip
else:
    print(f"Dataset ya existe en {RUTA_DATASET}, se omite la descarga.")


In [ ]:
df = pd.read_csv(RUTA_DATASET)
print(f"Registros cargados: {len(df):,}")
df.head(2)


## 3. Limpieza y preprocesamiento

A partir de las conclusiones del EDA, se aplican las siguientes decisiones:

| Hallazgo del EDA | Decisión de limpieza |
|---|---|
| `Company Profile` tiene 0.34% de nulos | Se rellenan con cadena vacía (no se usará para embeddings de todas formas) |
| Columnas de texto (`Job Title`, `Role`, `Qualifications`, `skills`, `Responsibilities`, `Job Description`) sin nulos relevantes, pero pueden tener espacios/ruido | Normalizar espacios en blanco, quitar saltos de línea repetidos |
| `Experience` es texto tipo `"5 to 8 Years"` | Extraer `experiencia_min` y `experiencia_max` numéricos para poder filtrar |
| `Salary Range` es texto tipo `"$59K-$99K"` | Extraer `salario_min` y `salario_max` numéricos (en miles) |
| `Job Posting Date` ya se parseó como fecha en el EDA | Se repite aquí por si se ejecuta el notebook de forma independiente |
| Columnas irrelevantes para búsqueda semántica (`Contact Person`, `Contact`, `Job Portal`) | Se descartan del dataset que alimentará el motor (se conservan solo si se necesitan para trazabilidad) |
| 0 registros duplicados, `Job Id` 100% único | No se requiere deduplicación adicional |


In [ ]:
COLUMNAS_TEXTO = [
    "Job Title",
    "Role",
    "Qualifications",
    "skills",
    "Responsibilities",
    "Job Description",
]

COLUMNAS_FILTROS = [
    "Experience",
    "Salary Range",
    "location",
    "Country",
    "Work Type",
    "Company Size",
    "Preference",
    "Job Posting Date",
]

COLUMNAS_IDENTIFICACION = ["Job Id", "Company"]

def limpiar_texto(texto: str) -> str:
    """Normaliza espacios en blanco y remueve caracteres de control."""
    if pd.isna(texto):
        return ""
    texto = str(texto)
    texto = re.sub(r"\s+", " ", texto)  # colapsa espacios/saltos de línea
    return texto.strip()

df_clean = df.copy()

for columna in COLUMNAS_TEXTO:
    df_clean[columna] = df_clean[columna].apply(limpiar_texto)

df_clean["Company Profile"] = df_clean["Company Profile"].fillna("")

print("Columnas de texto normalizadas.")


In [ ]:
# Parseo de Experience: "5 to 8 Years" -> experiencia_min=5, experiencia_max=8
# Version vectorizada (str.extract), mucho mas rapida que .apply() fila por fila sobre 1.6M registros.
extraido_experiencia = df_clean["Experience"].str.extract(
    r"(\d+)\s*to\s*(\d+)", flags=re.IGNORECASE
)

df_clean["experiencia_min"] = pd.to_numeric(extraido_experiencia[0])
df_clean["experiencia_max"] = pd.to_numeric(extraido_experiencia[1])

df_clean[["Experience", "experiencia_min", "experiencia_max"]].head()


In [ ]:
# Parseo de Salary Range: "$59K-$99K" -> salario_min=59, salario_max=99 (en miles USD)
# Version vectorizada, igual que con Experience.
extraido_salario = df_clean["Salary Range"].str.extract(
    r"\$?(\d+)K-\$?(\d+)K", flags=re.IGNORECASE
)

df_clean["salario_min"] = pd.to_numeric(extraido_salario[0])
df_clean["salario_max"] = pd.to_numeric(extraido_salario[1])

df_clean[["Salary Range", "salario_min", "salario_max"]].head()


In [ ]:
df_clean["Job Posting Date"] = pd.to_datetime(df_clean["Job Posting Date"], errors="coerce")

columnas_a_descartar = [c for c in ["Contact Person", "Contact", "Job Portal"] if c in df_clean.columns]
df_clean = df_clean.drop(columns=columnas_a_descartar)

print(f"Columnas descartadas: {columnas_a_descartar}")
print(f"Dimensiones tras limpieza: {df_clean.shape}")


### 3.1 Construcción del texto combinado para el embedding

Se concatenan los campos textuales relevantes en un único campo `texto_combinado`. Se repite `Job Title` y `skills` para darles un poco más de peso semántico, ya que suelen ser los campos más discriminativos para una búsqueda por habilidades/cargo.

In [ ]:
# Version vectorizada (str.cat), en lugar de .apply(axis=1) que es muy lento sobre 1.6M filas.
partes = [
    df_clean["Job Title"], df_clean["Job Title"],   # peso extra
    df_clean["Role"],
    df_clean["skills"], df_clean["skills"],          # peso extra
    df_clean["Qualifications"],
    df_clean["Responsibilities"],
    df_clean["Job Description"],
]

df_clean["texto_combinado"] = partes[0].str.cat(partes[1:], sep=" ").str.strip()

df_clean[["Job Title", "texto_combinado"]].head(2)


In [ ]:
# Se descartan registros sin ningún contenido textual útil (deberían ser 0 según el EDA, se valida igual)
antes = len(df_clean)
df_clean = df_clean[df_clean["texto_combinado"].str.len() > 0].reset_index(drop=True)
print(f"Registros removidos por texto vacío: {antes - len(df_clean):,}")
print(f"Registros finales: {len(df_clean):,}")


In [ ]:
RUTA_LIMPIO = RUTA_PROCESSED / "job_descriptions_clean.parquet"
df_clean.to_parquet(RUTA_LIMPIO, index=False)
print(f"Dataset limpio guardado en: {RUTA_LIMPIO}")
